# Notebook 02 - Lam sach du lieu Ha Noi

Muc tieu:
- Doc du lieu tho da enrich truong sau.
- Chon pham vi linh hoat: 1 quan hoac top-N quan de du so mau train.
- Lam sach missing, loai outlier, tao tap du lieu san sang train.
- Luu ra `data/processed/housing_hanoi_clean.csv`.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "housing_raw_vn_hanoi_deep.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH

WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/raw/housing_raw_vn_hanoi_deep.csv')

In [12]:
df = pd.read_csv(RAW_PATH)
print("Shape ban dau:", df.shape)
df.head(3)

Shape ban dau: (2879, 24)


,source,city,price_raw,area_raw,location_raw,detail_url,list_id,category_name,crawl_page,crawl_ts,...,chieuDai,Phongngu,PhongTam,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,DiaChi
0,chotot,Ha Noi,"19,8 tỷ",50.0,"Phường Láng Thượng, Ha Noi",https://cdn.chotot.com/Z4h1aUo9xoKjjsvboE08t0y...,131673394,Nhà ở,1,2026-05-01 18:14:13.228010+07:00,...,NaN,6.0,NaN,NaN,"Nhà ngõ, hẻm",NaN,NaN,Phường Láng Thượng,Quận Đống Đa,"Đường Nguyễn Chí Thanh, Phường Láng Thượng, Qu..."
1,chotot,Ha Noi,"6,35 tỷ",45.0,"Xã Hữu Hoà, Ha Noi",https://cdn.chotot.com/lTbplm1RmWfuOE6xkHLdeft...,130840917,Nhà ở,1,2026-05-01 18:14:13.230619+07:00,...,NaN,4.0,NaN,NaN,"Nhà mặt phố, mặt tiền",NaN,NaN,Xã Hữu Hoà,Huyện Thanh Trì,"Hữu lê, Xã Hữu Hoà, Huyện Thanh Trì, Hà Nội"
2,chotot,Ha Noi,6 tỷ,36.0,"Phường Hà Cầu, Ha Noi",https://cdn.chotot.com/2mDbOH9nNOXUNCDip0jto_N...,130728190,Nhà ở,1,2026-05-01 18:14:13.230619+07:00,...,NaN,4.0,NaN,NaN,"Nhà ngõ, hẻm",NaN,NaN,Phường Hà Cầu,Quận Hà Đông,"Bà triệu, Phường Hà Cầu, Quận Hà Đông, Hà Nội"


## 1) Chon 1 quan theo so luong mau

In [13]:
district_counts = df["Quan"].fillna("Khong ro").value_counts()
district_counts.head(15)

Quan
Quận Long Biên       357
Quận Hoàng Mai       353
Quận Hai Bà Trưng    347
Quận Đống Đa         316
Quận Thanh Xuân      249
Quận Hà Đông         213
Quận Cầu Giấy        205
Quận Nam Từ Liêm     150
Quận Bắc Từ Liêm     134
Quận Tây Hồ          109
Huyện Thanh Trì      102
Quận Ba Đình          98
Huyện Hoài Đức        85
Huyện Chương Mỹ       35
Huyện Đông Anh        33
Name: count, dtype: int64

In [14]:
# Luon chon duy nhat 1 quan co nhieu mau nhat (bo qua gia tri khong ro)
valid_district_counts = district_counts[district_counts.index != "Khong ro"]
TARGET_DISTRICT = valid_district_counts.index[0]
selected_districts = [TARGET_DISTRICT]

df1 = df[df["Quan"].isin(selected_districts)].copy()
print("Quan duoc chon:", TARGET_DISTRICT)
print("So mau trong quan:", len(df1))

Quan duoc chon: Quận Long Biên
So mau trong quan: 357


## 2) Chuan hoa kieu du lieu

In [15]:
num_cols = [
    "Gia", "dienTich", "chieuNgang", "chieuDai",
    "Phongngu", "PhongTam", "SoTang"
]

for c in num_cols:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

df1[num_cols].dtypes

Gia           float64
dienTich      float64
chieuNgang    float64
chieuDai      float64
Phongngu      float64
PhongTam      float64
SoTang        float64
dtype: object

## 3) Xu ly missing co ban

In [16]:
print("Missing truoc:\n", df1[num_cols + ["Loai", "GiayTo", "TinhTrangNoiThat"]].isna().sum())

Missing truoc:
 Gia                   0
dienTich              0
chieuNgang          147
chieuDai            198
Phongngu              0
PhongTam            119
SoTang               71
Loai                  0
GiayTo               71
TinhTrangNoiThat    147
dtype: int64


In [17]:
# Cac cot bat buoc toi thieu cho bai toan gia nha
df1 = df1.dropna(subset=["Gia", "dienTich"])

# Dien median cho cot so (giu lai nhieu mau hon)
for c in ["Phongngu", "PhongTam", "SoTang", "chieuNgang", "chieuDai"]:
    if c in df1.columns:
        df1[c] = df1[c].fillna(df1[c].median())

# Dien nhan 'Khong ro' cho cot phan loai
for c in ["Loai", "GiayTo", "TinhTrangNoiThat", "Phuong", "Quan"]:
    if c in df1.columns:
        df1[c] = df1[c].fillna("Khong ro")

print("Shape sau xu ly missing:", df1.shape)

Shape sau xu ly missing: (357, 24)


## 4) Loai outlier co ban

In [18]:
# Dieu kien hop ly so bo (noi rong de khong loai qua nhieu mau)
cond = (
    (df1["Gia"] >= 2e8) & (df1["Gia"] <= 5e11) &
    (df1["dienTich"] >= 8) & (df1["dienTich"] <= 1500) &
    (df1["Phongngu"] >= 0) & (df1["Phongngu"] <= 30) &
    (df1["PhongTam"] >= 0) & (df1["PhongTam"] <= 30) &
    (df1["SoTang"] >= 0) & (df1["SoTang"] <= 50)
)
df2 = df1[cond].copy()

df2["Gia_m2"] = df2["Gia"] / df2["dienTich"]
print("Shape sau loc outlier:", df2.shape)
df2[["Gia", "dienTich", "Gia_m2"]].describe()

Shape sau loc outlier: (356, 25)


,Gia,dienTich,Gia_m2
count,3.560000e+02,356.000000,3.560000e+02
mean,1.544952e+10,61.071065,2.506566e+08
std,1.218117e+10,39.651301,8.915705e+07
min,1.450000e+09,20.000000,3.625000e+07
25%,8.500000e+09,40.000000,1.926984e+08
50%,1.180000e+10,50.000000,2.343182e+08
75%,1.767500e+10,65.250000,2.963542e+08
max,9.000000e+10,312.000000,6.333333e+08


## 5) Chon cot dau vao train

In [20]:
final_cols = [
    "Gia", "Gia_m2", "dienTich", "chieuNgang", "chieuDai",
    "Phongngu", "PhongTam", "SoTang",
    "Loai", "GiayTo", "TinhTrangNoiThat", "Phuong", "Quan",
     "list_id"
]

final_cols = [c for c in final_cols if c in df2.columns]
df_clean = df2[final_cols].drop_duplicates(subset=["list_id"]).copy()
print("Shape cuoi:", df_clean.shape)
df_clean.head(3)

Shape cuoi: (356, 14)


,Gia,Gia_m2,dienTich,chieuNgang,chieuDai,Phongngu,PhongTam,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,list_id
21,9.000000e+09,2.571429e+08,35.0,350.0,10.0,3.0,4.0,5.0,"Nhà ngõ, hẻm",Đã có sổ,Khong ro,Phường Ngọc Lâm,Quận Long Biên,129628407
32,1.760000e+10,3.384615e+08,52.0,4.4,12.0,4.0,4.0,6.0,"Nhà mặt phố, mặt tiền",Đã có sổ,Nội thất cao cấp,Phường Long Biên,Quận Long Biên,130919547
34,6.500000e+09,2.124183e+08,30.6,7.0,4.3,3.0,4.0,5.0,"Nhà ngõ, hẻm",Đã có sổ,Nội thất đầy đủ,Phường Long Biên,Quận Long Biên,130571107


In [22]:
out_path = PROCESSED_DIR / "housing_hanoi_clean.csv"
df_clean.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Da luu du lieu clean tai: {out_path}")

Da luu du lieu clean tai: C:\Users\asus\Desktop\DuDoanGiaNha\data\processed\housing_hanoi_clean.csv
